#### 01 -- Order Items (Customer Transactions Table)
EDA: Schema, keys, nulls, duplicates, customer identification, prices and quantities

In [1]:
# Imports and setup

from pathlib import Path 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

project_root = Path.cwd().parent
raw_dir = project_root / "data" / "raw"

In [2]:
# Initialize Spark Session

spark = (
    SparkSession.builder
    .appName("CLVSourceProfiling")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/10 13:10:36 WARN Utils: Your hostname, SamDep, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/10 13:10:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/samdep/clv-analytics-pipeline/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/10 13:10:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Load order_items source data and conduct initial inspection

order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(raw_dir / "order_items.csv"))
)

order_items.printSchema()

order_items.show(10, truncate=False)

print(f"Total records: {order_items.count():,}")

# spark.stop()

root
 |-- APP_NAME: string (nullable = true)
 |-- RESTAURANT_ID: string (nullable = true)
 |-- CREATION_TIME_UTC: string (nullable = true)
 |-- ORDER_ID: string (nullable = true)
 |-- USER_ID: string (nullable = true)
 |-- PRINTED_CARD_NUMBER: long (nullable = true)
 |-- IS_LOYALTY: boolean (nullable = true)
 |-- CURRENCY: string (nullable = true)
 |-- LINEITEM_ID: string (nullable = true)
 |-- ITEM_CATEGORY: string (nullable = true)
 |-- ITEM_NAME: string (nullable = true)
 |-- ITEM_PRICE: double (nullable = true)
 |-- ITEM_QUANTITY: integer (nullable = true)

+--------------------------+------------------------+------------------------+------------------------+------------------------+-------------------+----------+--------+------------------------+-------------+-------------------------------------+----------+-------------+
|APP_NAME                  |RESTAURANT_ID           |CREATION_TIME_UTC       |ORDER_ID                |USER_ID                 |PRINTED_CARD_NUMBER|IS_LOYALTY|CU

Total records: 203,531


In [4]:
# Evaluate candidate keys by comparing total records,
# distinct order-item pairs, orders, and identified customers.

order_items.select(
    F.count("*").alias("total_records"),
    F.countDistinct(
        "ORDER_ID",
        "LINEITEM_ID"
    ).alias("distinct_order_items"),
    F.countDistinct(
        "ORDER_ID"
    ).alias("distinct_orders"),
    F.countDistinct(
        "USER_ID"
    ).alias("distinct_customers")
).show()

+-------------+--------------------+---------------+------------------+
|total_records|distinct_order_items|distinct_orders|distinct_customers|
+-------------+--------------------+---------------+------------------+
|       203531|              203518|         131328|             20174|
+-------------+--------------------+---------------+------------------+



In [5]:
# Check for duplicate (ORDER_ID, LINEITEM_ID) combinations.

duplicate_items = (
    order_items
    .groupBy("ORDER_ID", "LINEITEM_ID")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

duplicate_items.show(20, truncate=False)

+--------+-----------+-----+
|ORDER_ID|LINEITEM_ID|count|
+--------+-----------+-----+
|NULL    |NULL       |12   |
+--------+-----------+-----+



In [6]:
# Check whether any records have both ORDER_ID and LINEITEM_ID missing.

null_key_records = order_items.filter(
    F.col("ORDER_ID").isNull()
    & F.col("LINEITEM_ID").isNull()
)

null_key_records.show(20, truncate=False)

+--------+-------------+-----------------+--------+-------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+
|APP_NAME|RESTAURANT_ID|CREATION_TIME_UTC|ORDER_ID|USER_ID|PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|LINEITEM_ID|ITEM_CATEGORY|ITEM_NAME|ITEM_PRICE|ITEM_QUANTITY|
+--------+-------------+-----------------+--------+-------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+
|JUICE"  |1.99         |1                |NULL    |NULL   |NULL               |NULL      |NULL    |NULL       |NULL         |NULL     |NULL      |NULL         |
|JUICE"  |2.39         |1                |NULL    |NULL   |NULL               |NULL      |NULL    |NULL       |NULL         |NULL     |NULL      |NULL         |
|JUICE"  |1.99         |1                |NULL    |NULL   |NULL               |NULL      |NULL    |NULL       |NULL         |NULL     |NULL      |NULL         |
|JUICE"  |2.39         |1         

In [7]:
# Count non-NULL values in each column for records missing both key identifiers

null_key_records.select(
    [
        F.count(F.col(c)).alias(c)
        for c in order_items.columns
    ]
).show(truncate=False)

+--------+-------------+-----------------+--------+-------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+
|APP_NAME|RESTAURANT_ID|CREATION_TIME_UTC|ORDER_ID|USER_ID|PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|LINEITEM_ID|ITEM_CATEGORY|ITEM_NAME|ITEM_PRICE|ITEM_QUANTITY|
+--------+-------------+-----------------+--------+-------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+
|12      |12           |12               |0       |0      |0                  |0         |0       |0          |0            |0        |0         |0            |
+--------+-------------+-----------------+--------+-------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+



### Re-import with MultiLine is True

In [8]:
# Re-import with multiLine=True to test whether quoted multiline fields caused the record-count discrepancy

order_items_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv(str(raw_dir / "order_items.csv"))
)

print(f"Total records: {order_items_test.count():,}")

Total records: 203,519


In [9]:
# Validate record count, distinct order-item pairs, and missing ORDER_ID values after re-importing with multiLine=True

order_items_test.select(
    F.count("*").alias("total_records"),
    F.countDistinct(
        "ORDER_ID",
        "LINEITEM_ID"
    ).alias("distinct_order_items"),
    F.sum(
        F.col("ORDER_ID").isNull().cast("int")
    ).alias("null_order_ids")
).show()

+-------------+--------------------+--------------+
|total_records|distinct_order_items|null_order_ids|
+-------------+--------------------+--------------+
|       203519|              203518|             0|
+-------------+--------------------+--------------+



In [10]:
# Recheck for duplicate (ORDER_ID, LINEITEM_ID) combinations after correcting CSV parsing

order_items = order_items_test

duplicate_items = (
    order_items
    .groupBy("ORDER_ID", "LINEITEM_ID")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_items.show(truncate=False)

+--------+-----------+-----+
|ORDER_ID|LINEITEM_ID|count|
+--------+-----------+-----+
+--------+-----------+-----+



In [11]:
# Retrieve the original records for any duplicate (ORDER_ID, LINEITEM_ID) combinations

duplicate_records = order_items.join(
    duplicate_items.select("ORDER_ID", "LINEITEM_ID"),
    on=["ORDER_ID", "LINEITEM_ID"],
    how="inner"
)

duplicate_records.show(truncate=False)

+--------+-----------+--------+-------------+-----------------+-------+-------------------+----------+--------+-------------+---------+----------+-------------+
|ORDER_ID|LINEITEM_ID|APP_NAME|RESTAURANT_ID|CREATION_TIME_UTC|USER_ID|PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|ITEM_CATEGORY|ITEM_NAME|ITEM_PRICE|ITEM_QUANTITY|
+--------+-----------+--------+-------------+-----------------+-------+-------------------+----------+--------+-------------+---------+----------+-------------+
+--------+-----------+--------+-------------+-----------------+-------+-------------------+----------+--------+-------------+---------+----------+-------------+



In [12]:
# Check candidate key completeness by counting NULL ORDER_ID and LINEITEM_ID values

order_items.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("ORDER_ID").isNull().cast("int")).alias("null_order_ids"),
    F.sum(F.col("LINEITEM_ID").isNull().cast("int")).alias("null_lineitem_ids")
).show()

+-------------+--------------+-----------------+
|total_records|null_order_ids|null_lineitem_ids|
+-------------+--------------+-----------------+
|       203519|             0|                1|
+-------------+--------------+-----------------+



In [13]:
# Inspect records with a missing ORDER_ID or LINEITEM_ID to identify incomplete candidate keys

order_items.filter(
    F.col("ORDER_ID").isNull()
    | F.col("LINEITEM_ID").isNull()
).show(truncate=False)

+-------------+------------------------+-----------------------+------------------------+------------------------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+
|APP_NAME     |RESTAURANT_ID           |CREATION_TIME_UTC      |ORDER_ID                |USER_ID                 |PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|LINEITEM_ID|ITEM_CATEGORY|ITEM_NAME|ITEM_PRICE|ITEM_QUANTITY|
+-------------+------------------------+-----------------------+------------------------+------------------------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+
|Alltown Fresh|5e7e35ed902ad5ac017b242b|2021-12-07 11:23:30.917|61af8a82ffd125453e62ae37|609d681462e498b356e72a6d|NULL               |false     |USD     |NULL       |NULL         |NULL     |4.39      |0            |
+-------------+------------------------+-----------------------+------------------------+------------------------+-------------------+--

In [14]:
# Assess customer identifier completeness and count distinct identified customers

order_items.select(
    F.count("*").alias("total_records"),
    F.count("USER_ID").alias("records_with_user_id"),
    F.sum(
        F.col("USER_ID").isNull().cast("int")
    ).alias("missing_user_id"),
    F.countDistinct("USER_ID").alias("distinct_customers")
).show()

+-------------+--------------------+---------------+------------------+
|total_records|records_with_user_id|missing_user_id|distinct_customers|
+-------------+--------------------+---------------+------------------+
|       203519|              185711|          17808|             20174|
+-------------+--------------------+---------------+------------------+



In [15]:
# Calculate the percentage of order-item records with a missing USER_ID

order_items.select(
    F.round(
        F.avg(
            F.col("USER_ID").isNull().cast("int")
        ) * 100,
        2
    ).alias("missing_user_id_pct")
).show()

+-------------------+
|missing_user_id_pct|
+-------------------+
|               8.75|
+-------------------+



In [16]:
# Count NULL values across all columns to identify fields requiring data quality assessment

null_counts = order_items.select(
    [
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in order_items.columns
    ]
)

null_counts.show(truncate=False, vertical=True)

-RECORD 0---------------------
 APP_NAME            | 0      
 RESTAURANT_ID       | 0      
 CREATION_TIME_UTC   | 0      
 ORDER_ID            | 0      
 USER_ID             | 17808  
 PRINTED_CARD_NUMBER | 157435 
 IS_LOYALTY          | 0      
 CURRENCY            | 0      
 LINEITEM_ID         | 1      
 ITEM_CATEGORY       | 1      
 ITEM_NAME           | 1      
 ITEM_PRICE          | 0      
 ITEM_QUANTITY       | 0      



In [17]:
# Profile item prices and quantities, including zero and negative values

order_items.select(
    F.min("ITEM_PRICE").alias("min_price"),
    F.max("ITEM_PRICE").alias("max_price"),
    F.avg("ITEM_PRICE").alias("avg_price"),
    F.sum((F.col("ITEM_PRICE") < 0).cast("int")).alias("negative_prices"),
    F.sum((F.col("ITEM_PRICE") == 0).cast("int")).alias("zero_prices"),
    F.min("ITEM_QUANTITY").alias("min_quantity"),
    F.max("ITEM_QUANTITY").alias("max_quantity"),
    F.sum((F.col("ITEM_QUANTITY") < 0).cast("int")).alias("negative_quantities"),
    F.sum((F.col("ITEM_QUANTITY") == 0).cast("int")).alias("zero_quantities")
).show()

+---------+---------+----------------+---------------+-----------+------------+------------+-------------------+---------------+
|min_price|max_price|       avg_price|negative_prices|zero_prices|min_quantity|max_quantity|negative_quantities|zero_quantities|
+---------+---------+----------------+---------------+-----------+------------+------------+-------------------+---------------+
|      0.0|   5000.0|8.73988929778989|              0|        156|           0|         500|                  0|              1|
+---------+---------+----------------+---------------+-----------+------------+------------+-------------------+---------------+



In [18]:
# Inspect multi-quantity line items to investigate ITEM_PRICE semantics

order_items.filter(
    F.col("ITEM_QUANTITY") > 1
).select(
    "ORDER_ID",
    "LINEITEM_ID",
    "ITEM_NAME",
    "ITEM_PRICE",
    "ITEM_QUANTITY"
).show(20, truncate=False)

+------------------------+------------------------+-------------------------------------+----------+-------------+
|ORDER_ID                |LINEITEM_ID             |ITEM_NAME                            |ITEM_PRICE|ITEM_QUANTITY|
+------------------------+------------------------+-------------------------------------+----------+-------------+
|643a9c5cdc8b4326e0070364|643a9c5de7bc16bd7102398d|Southwest Sausage Breakfast Sandwich*|11.98     |2            |
|643eeab843345dfae40b0644|643eeade30831f68880ab516|Defense                              |18.0      |2            |
|6428889c00bb851f70053097|642888ddd8e26d4cb60816e0|Cheese Quesadilla                    |9.98      |2            |
|651829f8724e148549088e51|65182aed66096c565002dc21|Chilaquiles*                         |19.98     |2            |
|652445eb1d0f7691bb0de9dd|652446ee1dad32b58e0b8d02|Wagyu Barbacoa Bowl                  |35.97     |3            |
|6558c6be8eeebd97bd09e621|6558c6c0203ca18e9709c0db|ALLTOWN FRESH HOT COFFEE     

In [19]:
# Establish the transaction date range and identify unparseable timestamps

order_items.select(
    F.min(F.to_timestamp("CREATION_TIME_UTC")).alias("earliest_transaction"),
    F.max(F.to_timestamp("CREATION_TIME_UTC")).alias("latest_transaction"),
    F.sum(
        F.to_timestamp("CREATION_TIME_UTC").isNull().cast("int")
    ).alias("unparseable_timestamps")
).show(truncate=False)

+-----------------------+-----------------------+----------------------+
|earliest_transaction   |latest_transaction     |unparseable_timestamps|
+-----------------------+-----------------------+----------------------+
|2020-04-21 14:56:44.244|2024-02-20 21:52:35.366|0                     |
+-----------------------+-----------------------+----------------------+



In [20]:
# Examine loyalty status alongside customer and printed card identifier completeness

order_items.groupBy("IS_LOYALTY").agg(
    F.count("*").alias("total_records"),
    F.count("USER_ID").alias("records_with_user_id"),
    F.count("PRINTED_CARD_NUMBER").alias("records_with_card_number"),
    F.countDistinct("USER_ID").alias("distinct_identified_customers")
).show(truncate=False)

+----------+-------------+--------------------+------------------------+-----------------------------+
|IS_LOYALTY|total_records|records_with_user_id|records_with_card_number|distinct_identified_customers|
+----------+-------------+--------------------+------------------------+-----------------------------+
|true      |46084        |46056               |46084                   |5829                         |
|false     |157435       |139655              |0                       |17128                        |
+----------+-------------+--------------------+------------------------+-----------------------------+



In [21]:
# Check currency values and restaurant coverage before defining revenue aggregates

order_items.groupBy("CURRENCY").count().show(truncate=False)

order_items.select(
    F.countDistinct("RESTAURANT_ID").alias("distinct_restaurants")
).show()

+--------+------+
|CURRENCY|count |
+--------+------+
|USD     |203519|
+--------+------+

+--------------------+
|distinct_restaurants|
+--------------------+
|                  28|
+--------------------+



In [22]:
# Inspect outlier - extreme prices and quantities, determine if valid

order_items.filter(
    (F.col("ITEM_PRICE") == 5000)
    | (F.col("ITEM_QUANTITY") == 500)
).select(
    "ORDER_ID",
    "ITEM_NAME",
    "ITEM_CATEGORY",
    "ITEM_PRICE",
    "ITEM_QUANTITY"
).show(truncate=False)

+------------------------+-----------------------+-------------+----------+-------------+
|ORDER_ID                |ITEM_NAME              |ITEM_CATEGORY|ITEM_PRICE|ITEM_QUANTITY|
+------------------------+-----------------------+-------------+----------+-------------+
|5ed7972a4f5ee90547011c03|Espresso - Double      |Espresso     |1125.0    |500          |
|5ed7972a4f5ee90547011c03|Meet Your Matcha - 12oz|Smoothies    |3500.0    |500          |
|5f3d1ba7505ee9374b7b23cc|Korean Kimchi          |Bowls        |5000.0    |500          |
+------------------------+-----------------------+-------------+----------+-------------+



In [23]:
# Examine zero-price items by category and item name

order_items.filter(
    F.col("ITEM_PRICE") == 0
).groupBy(
    "ITEM_CATEGORY", "ITEM_NAME"
).count().orderBy(
    F.desc("count")
).show(20, truncate=False)

+-----------------------+-------------------------------------+-----+
|ITEM_CATEGORY          |ITEM_NAME                            |count|
+-----------------------+-------------------------------------+-----+
|Breakfast              |Southwest Sausage Breakfast Sandwich*|12   |
|Breakfast              |Fresh Classic* | VEG                 |12   |
|Meal Prep Kits         |Tomato Soup & Grilled Cheese         |10   |
|Drip Coffee            |ALLTOWN FRESH HOT COFFEE             |9    |
|Sandwiches             |Smoked B.L.A.T                       |8    |
|Salads                 |Kale Caesar Salad                    |7    |
|Sandwiches             |Chipotle Turkey                      |6    |
|Breakfast              |Smoked Ham & Hot Honey Egg & Cheese* |4    |
|Soda, Juices & Kombucha|SARSAPARILA SODA                     |4    |
|Salads                 |Farmer's Market | V, VEG, GF         |3    |
|Salads                 |Chop Salad                           |3    |
|Sandwiches         

In [24]:
# Check whether card numbers on loyalty records missing USER_ID
# also appear on records with an identified customer

missing_loyalty_users = (
    order_items
    .filter(
        (F.col("IS_LOYALTY") == True)
        & F.col("USER_ID").isNull()
    )
    .select("PRINTED_CARD_NUMBER")
    .distinct()
)

known_card_users = (
    order_items
    .filter(F.col("USER_ID").isNotNull())
    .groupBy("PRINTED_CARD_NUMBER")
    .agg(
        F.countDistinct("USER_ID").alias("distinct_users_per_card")
    )
)

missing_loyalty_users.join(
    known_card_users,
    on="PRINTED_CARD_NUMBER",
    how="left"
).groupBy("distinct_users_per_card").count().show()

+-----------------------+-----+
|distinct_users_per_card|count|
+-----------------------+-----+
|                   NULL|    4|
|                      1|   14|
+-----------------------+-----+

